In [ ]:
CATALOG = "spotify_etl"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
print("Target: spotify_etl.silver.dim_time")

In [ ]:
MERGE INTO spotify_etl.silver.dim_time t
USING (
  SELECT DISTINCT
    to_timestamp(played_at) AS play_timestamp,
    hour(to_timestamp(played_at)) AS hour_of_day,
    dayofweek(to_timestamp(played_at)) - 1 AS day_of_week,
    date_format(to_timestamp(played_at), 'EEEE') AS weekday_name,
    dayofweek(to_timestamp(played_at)) > 5 AS is_weekend,
    cast(quarter(to_timestamp(played_at)) AS STRING) AS quarter,
    year(to_timestamp(played_at)) AS year,
    month(to_timestamp(played_at)) AS month,
    day(to_timestamp(played_at)) AS day
  FROM spotify_etl.bronze.bronze_play_history
  WHERE played_at IS NOT NULL
) s ON t.play_timestamp = s.play_timestamp
WHEN NOT MATCHED THEN INSERT *


In [ ]:
result = spark.table(f"{CATALOG}.silver.dim_time")
print(f"Table state: {result.count()} rows")